In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from textblob import TextBlob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import contractions
import spacy  
import string
import re 

## **1) Data Exraction**

In [80]:
data = PyPDFLoader('AIML_PDF.pdf').load()

In [81]:
data1 = ""
for i in range(len(data)):
    data1 += data[i].page_content
data = data1

In [82]:
print(data)

AI & Machine Learning — Topic-Wise Guide
Page 2
CHAPTER 1: Introduction to AI and Intelligent Systems
Understand the field before choosing algorithms.
Artificial Intelligence (AI) is the discipline of building systems that perform tasks associated with human intelligence, including perception,
language understanding, reasoning, planning, and decision support. AI may use rules, search, optimization, machine learning, or combinations of
these.
1.1 AI, ML, Deep Learning, and Generative AI
  Artificial Intelligence: The umbrella field for systems that exhibit intelligent behaviour.
  Machine Learning (ML): A branch of AI in which models learn patterns from data rather than relying only on manually written rules.
  Deep Learning (DL): ML using neural networks with multiple layers; useful for complex image, audio, and text patterns.
  Generative AI: Models that generate content such as text, images, audio, code, or video.
 1.2 Types and applications
  Narrow AI: Built for a particular t

## **2) Text Normalization**

In [83]:
nlp = spacy.load("en_core_web_sm")
data = data.lower()
tokens = nlp(data) 

In [84]:
updated_tokens = [str(token) for token in tokens if not token.is_stop ]
data = " ".join(updated_tokens)

In [85]:
var = string.punctuation
data = re.sub(f"[{var}]","",data)

In [86]:
data = re.sub(r"[0-9]", "", data)

In [87]:
data = data.replace('page',"")

In [88]:
data = re.sub(r"\s{2,}"," ",data)
type(data)

str

In [89]:
data = contractions.fix(data)
data = TextBlob(data).correct() 
data = str(data)

In [90]:
print(data)

ai machine learning — topic wise guide chapter introduction ai intelligent systems understand field choosing algorithms artificial intelligence ai discipline building systems perform tasks associated human intelligence including perception language understanding reasoning planning decision support ai use rules search optimization machine learning combinations ai my deep learning generative ai  artificial intelligence umbrella field systems exhibit intelligent behaviour  machine learning my branch ai models learn patterns data relying mentally written rules  deep learning do my neutral network multiple layers useful complex image audit text patterns  generative ai models generate content text images audit code video types applications  narrow ai built particular task deployed ai today fits category general ai remains hypothetical broadly human level system  rule based systems apply explicit conditions learning based systems infer patterns examples  applications include fraud dete

## **3) Chuncking**

In [91]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50,
    separators = ['\n\n','.','\n',' ','']
)

In [92]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 100,
    separators=['\n\n','.','!','\n',' ','']
)
chunks = splitter.create_documents([str(data)])
for chunk in chunks:
    print(chunk.page_content)

ai machine learning — topic wise guide chapter introduction ai intelligent systems understand field choosing algorithms artificial intelligence ai discipline building systems perform tasks associated
choosing algorithms artificial intelligence ai discipline building systems perform tasks associated human intelligence including perception language understanding reasoning planning decision support
human intelligence including perception language understanding reasoning planning decision support ai use rules search optimization machine learning combinations ai my deep learning generative ai 
ai use rules search optimization machine learning combinations ai my deep learning generative ai  artificial intelligence umbrella field systems exhibit intelligent behaviour  machine learning my
artificial intelligence umbrella field systems exhibit intelligent behaviour  machine learning my branch ai models learn patterns data relying mentally written rules  deep learning do my neutral
branch a

In [93]:
chunks[0].metadata={'data':'AIML_PDF.pdf'}

In [94]:
chunks

[Document(metadata={'data': 'AIML_PDF.pdf'}, page_content='ai machine learning — topic wise guide chapter introduction ai intelligent systems understand field choosing algorithms artificial intelligence ai discipline building systems perform tasks associated'),
 Document(metadata={}, page_content='choosing algorithms artificial intelligence ai discipline building systems perform tasks associated human intelligence including perception language understanding reasoning planning decision support'),
 Document(metadata={}, page_content='human intelligence including perception language understanding reasoning planning decision support ai use rules search optimization machine learning combinations ai my deep learning generative ai \x7f'),
 Document(metadata={}, page_content='ai use rules search optimization machine learning combinations ai my deep learning generative ai \x7f artificial intelligence umbrella field systems exhibit intelligent behaviour \x7f machine learning my'),
 Document(meta

## **4) Embedding**

In [98]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## **5)Storing in VectorDB**

In [101]:
vectordb = FAISS.from_documents(
    documents = chunks,
    embedding = embedding_model
)
vectordb

In [103]:
user_query = "What is Machine Learning ?"
ret_chunks = vectordb.similarity_search(user_query)
ret_chunks

[Document(id='8b376925-0855-4756-a35f-07f6d0a7e99e', metadata={'data': 'AIML_PDF.pdf'}, page_content='ai machine learning — topic wise guide chapter introduction ai intelligent systems understand field choosing algorithms artificial intelligence ai discipline building systems perform tasks associated'),
 Document(id='d8146fa3-c6b0-4687-9def-308ebe36c2f0', metadata={}, page_content='artificial intelligence umbrella field systems exhibit intelligent behaviour \x7f machine learning my branch ai models learn patterns data relying mentally written rules \x7f deep learning do my neutral'),
 Document(id='561cf421-153d-4402-b5c9-e77af28207f7', metadata={}, page_content='learning uses labelled examples repression predict numerical values classification predict classes categories common algorithms \x7f linear repression predict numerical target weighted features useful'),
 Document(id='baec1b51-2ecd-4a97-bcd7-751119d5eca0', metadata={}, page_content='ai use rules search optimization machine lear

In [107]:
ret_data = ""
for chunk in ret_chunks:
    ret_data += chunk.page_content
ret_data

'ai machine learning — topic wise guide chapter introduction ai intelligent systems understand field choosing algorithms artificial intelligence ai discipline building systems perform tasks associatedartificial intelligence umbrella field systems exhibit intelligent behaviour \x7f machine learning my branch ai models learn patterns data relying mentally written rules \x7f deep learning do my neutrallearning uses labelled examples repression predict numerical values classification predict classes categories common algorithms \x7f linear repression predict numerical target weighted features usefulai use rules search optimization machine learning combinations ai my deep learning generative ai \x7f artificial intelligence umbrella field systems exhibit intelligent behaviour \x7f machine learning my'

## **Retrieval**

In [114]:
def retrieval(query : str,k):
    r_chunk = vectordb.similarity_search(query)
    retrieved_data = ""
    for chunk in r_chunk:
        retrieved_data += chunk.page_content
    return retrieved_data
user_query = "What is Deep Learning"
try :
    retrieval(user_query,4)
except Exception as e:
    print(f"Error : {e}")
finally:
    print(retrieval(user_query,4))

 transformer uses attention model relationships tokens supports parallel training  autoencoder learns compressed representations reconstructed input training challenges network overfat requiredetection recommendation demand forecasting medical image support translation document processing prediction maintenance assistants key idea ai overall field my approach ai deep learning sunset mybranch ai models learn patterns data relying mentally written rules  deep learning do my neutral network multiple layers useful complex image audit text patterns  generative ai models generategradients  optimizer sad adam update parameter epoch pass data batch sunset learning rate controls update size architecture  map feed forward network vector tubular input  can learns spatial


## ***Structuring Output***

In [121]:
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

In [124]:
def generate_text(query,answer):
    prompt = f'''
        You are my AI Assistant,
        Task : Structure the Output,Without Extra Content just with the retrived content {answer}
        Input : {query}
        Output : {answer}
        Note : 
            1)don't add any extra content.
            2)Whatever there in my {answer} just structure it.         
            Output Format:
            Input : {query}
            Output :
        '''
    gemini = gemini_model.invoke(prompt)
    print(gemini.content[0]['text'])
user_query = "What are Common Machine Learning Algorithms ?"
generate_text(user_query,retrieval(user_query,4))

Input : What are Common Machine Learning Algorithms ?
Output : 

### AI & Machine Learning — Topic-Wise Guide

#### 1. Chapter Introduction: AI & Intelligent Systems
* **Artificial Intelligence**: An umbrella field where systems exhibit intelligent behaviour; a discipline focused on building systems to perform tasks.
* **Goal**: Understand the field and choosing algorithms.

#### 2. System Types
* **Rule-Based Systems**: Apply explicit conditions.
* **Learning-Based Systems**: Infer patterns from examples.

#### 3. Machine Learning Types & Tasks
* Uses labelled examples.
* **Regression**: Predict numerical values.
* **Classification**: Predict classes and categories.

#### 4. Common Algorithms
* **Linear Regression**: Predict numerical target using weighted features (useful / fits category).

#### 5. General AI
* General AI remains hypothetical (broadly human-level system).

#### 6. Applications & Subfields
* **Applications**: Include fraud prevention/detection.
* **Techniques**: AI us